# 🏈 Predictor NFL v4

Agente de predicción de juegos NFL sobre datos [nflverse](https://github.com/nflverse). Todo el pipeline vive en **`nfl_pred.py`** — este notebook es la interfaz interactiva.

**Predice por juego:** puntos por equipo, totales, spread, probabilidad de victoria calibrada.
**Props por jugador:** yardas (pase/tierra/recepción) con rango propio por jugador, **TDs** (esperado + prob de ≥1), **recepciones** e **intercepciones**.

**Modelo dual:**
- `modelo_vegas` (usa spread/total de Vegas) → marcadores finos
- `modelo_puro` (sin Vegas) → opinión propia; cuando discrepa >4 pts de Vegas se marca **`value`**

**Features:** forma últimos 5 juegos (puntos, yardas, EPA of/def), localía, descanso, juego divisional, clima (domo/temp/viento), líneas de Vegas. **Props además usan:** % de snaps jugados y Next Gen Stats (CPOE del QB, separación del receptor, yardas sobre lo esperado del corredor).

**Modelo de puntos = ensamble** XGBoost + LightGBM + Ridge. **Probabilidad re-calibrada automáticamente:** los juegos ya jugados de la temporada actual entran solos a la calibración cada corrida.

**Automatización:** GitHub Actions corre `actualizar.py` cada martes, commitea `predicciones.csv` y publica el reporte web (`docs/index.html` → GitHub Pages).

## 1. Inicializar (carga datos + entrena)

Primera vez descarga ~300 MB (después cachea). `walk_forward=True` agrega el backtest honesto (re-entrena semana a semana de 2025, ~1-2 min extra).

In [1]:
from nfl_pred import *

ctx = inicializar(walk_forward=True)

Aviso: reportes de lesion aun no publicados (salen con la temporada)
Juegos: 1693 | Jugador-semanas: 112447 | Jugadas pbp: 294989 | Snaps: 157439 | NGS: 14909
Calendario 2026: 272 juegos | Roster: 2930 | Depth chart: 3172


Tabla equipo-juego: 3386 filas | matriz modelo: 3322 filas


--- Backtest 2025 (split simple, ensamble XGB+LGBM+Ridge) ---
MAE puntos por equipo: 7.38 | MAE total juego: 10.67
Acierto ganador - modelo puro: 62.5% | modelo c/Vegas: 63.5% | linea Vegas: 65.6% | prob calibrada: 67.0%


MAE yardas_pase: 72.52  (664 jugador-juegos)


MAE yardas_tierra: 17.07  (2320 jugador-juegos)


MAE yardas_recepcion: 16.22  (5476 jugador-juegos)
MAE td_pase: 0.93  (664 jugador-juegos)


MAE td_tierra: 0.31  (2320 jugador-juegos)


MAE td_recepcion: 0.23  (5476 jugador-juegos)


MAE recepciones: 1.23  (5476 jugador-juegos)
MAE intercepciones: 0.67  (664 jugador-juegos)


--- Backtest walk-forward 2025 (285 juegos, re-entrenando cada semana) ---
MAE puntos: 7.42 | acierto puro: 63.2% | c/Vegas: 66.7% | linea Vegas: 65.6%


**Cómo leer las métricas:**
- *split simple* = entrena 2020–2024, predice 2025 completo de una vez
- *walk-forward* = re-entrena cada semana con lo disponible hasta entonces (así funcionará en 2026 real)
- *prob calibrada* = regresión logística sobre `[diferencial del modelo puro, spread de Vegas]`; calibrada in-sample sobre 2025 — tómala como aproximación
- La línea de Vegas es el techo práctico (~66%); el valor del modelo está en las **discrepancias** y las **props**

## 2. Predecir un juego

Busca el juego en el calendario oficial 2026 (líneas de Vegas, descanso, clima/domo, división). Rosters y depth charts oficiales; excluye lesionados cuando hay reporte.

```python
predecir_juego('SEA', 'NE')          # visitante NE @ local SEA
predecir_juego('KC', 'BUF', week=5)  # semana específica
```

In [2]:
props = predecir_juego('SEA', 'NE')
props

Juego oficial: semana 1 (2026-09-09) | Vegas: spread +3.5, total 44.5
=== NE @ SEA ===
Marcador predicho:  SEA 26.0 - NE 19.8
Puntos totales:     45.8
Modelo puro (sin Vegas): SEA por +7.3 | discrepancia vs Vegas: +3.8
Prob. victoria (calibrada): SEA 68% | NE 32%


  Sin historial NFL (SEA): Jadarian Price (RB)


  Sin historial NFL (NE): Eli Raridon (TE)


,equipo,jugador,pos,prop,prediccion,rango_68pct,prob_1mas
0,NE,Drake Maye,QB,intercepciones,0.42,NaN,0.34
1,NE,A.J. Brown,WR,recepciones,4.91,NaN,0.99
2,NE,Romeo Doubs,WR,recepciones,3.26,NaN,0.96
3,NE,Hunter Henry,TE,recepciones,2.89,NaN,0.94
4,NE,Kayshon Boutte,WR,recepciones,2.83,NaN,0.94
5,NE,Rhamondre Stevenson,RB,recepciones,2.73,NaN,0.93
6,NE,TreVeyon Henderson,RB,recepciones,1.17,NaN,0.69
7,NE,Drake Maye,QB,td_pase,1.19,NaN,0.70
8,NE,A.J. Brown,WR,td_recepcion,0.37,NaN,0.31
9,NE,Romeo Doubs,WR,td_recepcion,0.26,NaN,0.23


**Columnas de props:**
- `rango_68pct` — rango propio del jugador (quantile regression: jugadores estables → rango angosto)
- `prob_1mas` — probabilidad de al menos 1 TD / 1 intercepción (Poisson)

## 3. Predecir una semana completa

`discrepancia` = spread del modelo puro − spread de Vegas. `value=True` cuando |discrepancia| > 4 pts: juegos donde el modelo "ve algo" distinto al mercado.

In [3]:
semana = predecir_semana(1)
semana

,season,week,fecha,local,visitante,pred_local,pred_visita,total_pred,total_vegas,spread_pred,spread_puro,spread_vegas,discrepancia,value,ganador,prob
0,2026,1,2026-09-09,SEA,NE,26.0,19.8,45.8,44.5,6.2,7.3,3.5,3.8,False,SEA,0.68
1,2026,1,2026-09-10,LA,SF,26.3,23.9,50.2,48.5,2.4,3.9,3.0,0.9,False,LA,0.62
2,2026,1,2026-09-13,CAR,CHI,21.5,22.3,43.8,44.5,-0.7,-0.4,-2.5,2.1,False,CHI,0.60
3,2026,1,2026-09-13,CIN,TB,27.0,24.9,51.9,50.5,2.1,4.1,3.5,0.6,False,CIN,0.63
4,2026,1,2026-09-13,DET,NO,28.3,24.6,52.8,48.5,3.7,-6.5,7.0,-13.5,True,DET,0.56
5,2026,1,2026-09-13,HOU,BUF,22.7,24.0,46.8,45.5,-1.3,-0.6,-1.5,0.9,False,BUF,0.58
6,2026,1,2026-09-13,IND,BAL,22.1,28.0,50.1,49.5,-5.9,-9.5,-3.5,-6.0,True,BAL,0.75
7,2026,1,2026-09-13,JAX,CLE,27.0,17.9,44.9,40.5,9.0,13.2,7.5,5.7,True,JAX,0.83
8,2026,1,2026-09-13,PIT,ATL,22.1,18.9,40.9,42.5,3.2,3.5,3.0,0.5,False,PIT,0.61
9,2026,1,2026-09-13,TEN,NYJ,19.7,18.6,38.3,39.5,1.1,8.9,3.0,5.9,True,TEN,0.69


In [4]:
# solo los juegos con posible value
semana[semana['value']]

,season,week,fecha,local,visitante,pred_local,pred_visita,total_pred,total_vegas,spread_pred,spread_puro,spread_vegas,discrepancia,value,ganador,prob
4,2026,1,2026-09-13,DET,NO,28.3,24.6,52.8,48.5,3.7,-6.5,7.0,-13.5,True,DET,0.56
6,2026,1,2026-09-13,IND,BAL,22.1,28.0,50.1,49.5,-5.9,-9.5,-3.5,-6.0,True,BAL,0.75
7,2026,1,2026-09-13,JAX,CLE,27.0,17.9,44.9,40.5,9.0,13.2,7.5,5.7,True,JAX,0.83
9,2026,1,2026-09-13,TEN,NYJ,19.7,18.6,38.3,39.5,1.1,8.9,3.0,5.9,True,TEN,0.69
10,2026,1,2026-09-13,LAC,ARI,25.8,21.2,47.0,45.5,4.5,1.4,11.5,-10.1,True,LAC,0.78
11,2026,1,2026-09-13,LV,MIA,23.3,21.5,44.8,41.5,1.8,-2.1,3.0,-5.1,True,LV,0.52
12,2026,1,2026-09-13,MIN,GB,20.8,24.4,45.2,44.5,-3.6,9.1,-1.5,10.6,True,MIN,0.58
14,2026,1,2026-09-13,NYG,DAL,22.0,24.2,46.2,48.5,-2.2,3.1,-2.5,5.6,True,DAL,0.54


## 4. Registro de predicciones

- `guardar_predicciones(week)` — anexa a `predicciones.csv` (sin duplicar semanas)
- `evaluar_predicciones()` — acierto real contra resultados y contra Vegas

En temporada esto lo hace solo GitHub Actions cada martes; también puedes correrlo a mano.

In [5]:
guardar_predicciones(1)
evaluar_predicciones()

16 juegos de semana 1 guardados en predicciones.csv
Aun no hay resultados para las predicciones guardadas


### Reporte web

`generar_reporte()` escribe `docs/index.html` con la última semana guardada (marcadores, totales y spreads vs Vegas, discrepancias con badge VALUE, historial de acierto). GitHub Actions lo regenera cada martes; con GitHub Pages activado (Settings → Pages → main, carpeta /docs) queda publicado en la web.

In [6]:
generar_reporte()

Reporte de semana 1 escrito en docs\index.html


'docs\\index.html'

## 5. Notas

**Flujo semanal automático (GitHub Actions):** martes 14:00 UTC corre `actualizar.py` → predice la semana próxima → commitea `predicciones.csv`. Disparo manual: pestaña *Actions* → *Predicciones semanales* → *Run workflow*.

**Flujo manual:** `nfl.clear_cache()` → re-ejecutar este notebook → `guardar_predicciones(week)`.

**Limitaciones honestas:**
- En offseason la "forma" viene de fin de 2025; los rosters ya son 2026 pero los números individuales son del equipo anterior del jugador. Se corrige solo al correr semanas de 2026 (el pipeline las incluye automáticamente cuando existen).
- Novatos sin historial NFL se omiten en props (se avisa).
- La probabilidad calibrada se ajustó sobre el mismo backtest 2025 (in-sample); re-calibrar tras unas semanas de 2026.